In [53]:
import pandas as pd
from collections import Counter
import numpy as np

# data

In [54]:
df1=pd.read_csv("C:/Users/user/Downloads/reviews_Porta Portese_classified.csv")
df2=pd.read_csv("C:/Users/user/Downloads/reviews_萬神殿_classified.csv")
df3=pd.read_csv("C:/Users/user/Downloads/reviews_萬神殿2_classified.csv")
df_all=pd.concat([df1, df2, df3], ignore_index=True)


path1="C:/Users/user/Downloads/reviews_Porta Portese_clean.csv"
path2="C:/Users/user/Downloads/reviews_萬神殿_clean.csv"
path3="C:/Users/user/Downloads/reviews_萬神殿2_clean.csv"
path_all="C:/Users/user/Downloads/mining_data_clean.csv"


df_raw=df_all
output_path=path_all

# 整理函式

In [55]:
def clean_df(df):
    rating=df["rating"].fillna("EMPTY").astype(str).str.strip().replace("","EMPTY")
    print(rating.value_counts().sort_index(ascending=False), end="\n\n")

    star=[int(s[0]) for s in rating]
    print("star")
    for key, value in sorted(Counter(star).items(), reverse=True):
        print(str(key)+": "+str(value))
    print()
    prdict_sentiment=df["sentiment"].fillna("EMPTY").astype(str).str.strip().replace("","EMPTY")
    for idx, i in enumerate(prdict_sentiment):
        if i=="正":
            prdict_sentiment[idx]="正面"
        elif i=="負":
            prdict_sentiment[idx]="負面"
        elif i=="中":
            prdict_sentiment[idx]="中立"

    print(prdict_sentiment.value_counts(), end="\n\n")

    matrix=[[] for _ in range(9)]

    for idx, score in enumerate(star):
        if (score==4 or score==5) and prdict_sentiment[idx]=="正面":
            matrix[0].append(idx)
        elif (score==3) and prdict_sentiment[idx]=="正面":
            matrix[1].append(idx)
        elif (score==1 or score==2) and prdict_sentiment[idx]=="正面":
            matrix[2].append(idx)
        elif (score==4 or score==5) and prdict_sentiment[idx]=="中立":
            matrix[3].append(idx)
        elif (score==3) and prdict_sentiment[idx]=="中立":
            matrix[4].append(idx)
        elif (score==1 or score==2) and prdict_sentiment[idx]=="中立":
            matrix[5].append(idx)
        elif (score==4 or score==5) and prdict_sentiment[idx]=="負面":
            matrix[6].append(idx)
        elif (score==3) and prdict_sentiment[idx]=="負面":
            matrix[7].append(idx)
        elif (score==1 or score==2) and prdict_sentiment[idx]=="負面":
            matrix[8].append(idx)

    row_label=["正面", "中立", "負面"]
    col_label=["4~5星", "3星", "1~2星"]

    matrix_count=np.array([len(a) for a in matrix]).reshape(3, 3)

    print(" \t"+"\t".join(col_label))
    for r, label in enumerate(row_label):
        print(f"{label:<4}\t" + "\t".join(map(str, matrix_count[r])))

    df["rating"]=star
    df["sentiment"]=prdict_sentiment
    df=df.drop(columns=["reviewer", "title"])
    return df, matrix_count

In [56]:
def date_sort(df):
    date=df["date"].str.split("•", n=1, expand=True)
    df["date_raw"]=date[0].str.strip()
    df["label"]=date[1].str.strip()

    parsed= pd.to_datetime(df["date_raw"], format="%b %Y", errors="coerce")
    df["date_status"]="valid"
    df.loc[parsed.isna(), "date_status"] = "unknown_format"
    df.loc[df["label"].notna(), "date_status"] = "date_with_label"
    df.loc[(df["label"].isna()) & (parsed.notna()), "date_status"] = "date_only"


    temp=df[df["date_status"]=="unknown_format"]
    print("unknown_format: "+str(len(temp)))
    temp=df[df["date_status"]=="date_with_label"]
    print("date_with_label: "+str(len(temp)))
    temp=df[df["date_status"]=="date_only"]
    print("date_only: "+str(len(temp)))

    df=df.drop(columns=["date", "trip_type"])
    df=df.rename(columns={"date_raw": "date", "label": "trip_type"})

    return df

In [57]:
print(df1["date"].head())

0    Mar 2026 • Couples
1                26-Mar
2                26-Feb
3    Feb 2026 • Couples
4                26-Jan
Name: date, dtype: object


# 整理與確認csv資料

In [58]:
df_clean, matrix_count=clean_df(df_raw)

rating
5 of 5 bubbles    5505
4 of 5 bubbles    1397
3 of 5 bubbles     394
2 of 5 bubbles     113
1 of 5 bubbles     217
Name: count, dtype: int64

star
5: 5505
4: 1397
3: 394
2: 113
1: 217

sentiment
正面    6258
中立     709
負面     659
Name: count, dtype: int64

 	4~5星	3星	1~2星
正面  	6172	82	4
中立  	554	142	13
負面  	176	170	313


In [59]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7626 entries, 0 to 7625
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   rating     7626 non-null   int64  
 1   text       7626 non-null   object 
 2   date       7626 non-null   object 
 3   trip_type  0 non-null      float64
 4   sentiment  7626 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 298.0+ KB


In [60]:
df_new=date_sort(df_clean)

unknown_format: 1561
date_with_label: 4576
date_only: 1489


In [61]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7626 entries, 0 to 7625
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   rating       7626 non-null   int64 
 1   text         7626 non-null   object
 2   sentiment    7626 non-null   object
 3   date         7626 non-null   object
 4   trip_type    4576 non-null   object
 5   date_status  7626 non-null   object
dtypes: int64(1), object(5)
memory usage: 357.6+ KB


In [62]:
df_new.to_csv(output_path, index=False, encoding="utf-8-sig")

# print

In [63]:
row_label=["正面", "中立", "負面"]
col_label=["4~5星", "3星", "1~2星"]

print(" \t"+"\t".join(col_label))
for r, label in enumerate(row_label):
    print(f"{label:<4}\t" + "\t".join(map(str, matrix_count[r])))

 	4~5星	3星	1~2星
正面  	6172	82	4
中立  	554	142	13
負面  	176	170	313


In [64]:
temp=df_new[df_new["date_status"]=="unknown_format"]
print("日期自動解讀失敗: "+str(len(temp)))
temp=df_new[df_new["date_status"]=="date_with_label"]
print("date_with_label: "+str(len(temp)))
temp=df_new[df_new["date_status"]=="date_only"]
print("date_only: "+str(len(temp)))

日期自動解讀失敗: 1561
date_with_label: 4576
date_only: 1489


In [65]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7626 entries, 0 to 7625
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   rating       7626 non-null   int64 
 1   text         7626 non-null   object
 2   sentiment    7626 non-null   object
 3   date         7626 non-null   object
 4   trip_type    4576 non-null   object
 5   date_status  7626 non-null   object
dtypes: int64(1), object(5)
memory usage: 357.6+ KB
